# Notebook 04 — Web Deployment

Convert the best trained model to TensorFlow.js format and deploy the real-time web interface.

**Prerequisites:** Complete Notebook 02 (model training) before running this notebook.


In [ ]:
import sys, os
sys.path.insert(0, '..')

import tensorflow as tf
import numpy as np
import json

from src.data.loader  import ASLDataLoader
from src.evaluation.metrics import ModelEvaluator
from src.utils.config import load_config

cfg = load_config('../config.yaml')
print('TensorFlow version:', tf.__version__)

## 1. Choose Best Model for Deployment

In [ ]:
# Load test data to pick the best model
loader = ASLDataLoader(data_dir='../data/raw')
_, _, X_test, _, _, y_test = loader.load_dataset()

candidate_paths = {
    'custom_cnn'   : '../results/models/custom_cnn_best.h5',
    'mobilenet'    : '../results/models/mobilenet_best.h5',
    'efficientnet' : '../results/models/efficientnet_best.h5',
}

# For web, prefer MobileNetV2 — best accuracy/size trade-off
deploy_model_name = 'mobilenet'
deploy_model_path = candidate_paths[deploy_model_name]
print(f'Deploying: {deploy_model_name} from {deploy_model_path}')

## 2. Convert to TensorFlow.js

In [ ]:
import subprocess

output_dir = '../web/tfjs_model'
os.makedirs(output_dir, exist_ok=True)

result = subprocess.run([
    'tensorflowjs_converter',
    '--input_format=keras',
    '--output_format=tfjs_graph_model',
    '--quantize_float16',      # Reduce model size by ~2x
    deploy_model_path,
    output_dir
], capture_output=True, text=True)

print('STDOUT:', result.stdout)
print('STDERR:', result.stderr)
print('Return code:', result.returncode)

if result.returncode == 0:
    files = os.listdir(output_dir)
    total_size = sum(os.path.getsize(os.path.join(output_dir, f)) for f in files)
    print(f'\nConverted successfully. Files: {files}')
    print(f'Total model size: {total_size / 1024 / 1024:.2f} MB')
else:
    print('Conversion failed. Make sure tensorflowjs is installed:')
    print('  pip install tensorflowjs')

## 3. Export Class Labels

In [ ]:
labels_path = '../web/labels.json'
with open(labels_path, 'w') as f:
    json.dump(loader.class_names, f)
print(f'Labels saved to {labels_path}')
print(f'Classes: {loader.class_names}')

## 4. Verify Conversion (Quick Sanity Check)

In [ ]:
original_model = tf.keras.models.load_model(deploy_model_path)
preds = np.argmax(original_model.predict(X_test[:20], verbose=0), axis=1)
accuracy = np.mean(preds == y_test[:20])
print(f'Sanity check accuracy on 20 samples: {accuracy:.2%}')


## 5. Model Size Comparison

In [ ]:
import matplotlib.pyplot as plt

model_sizes = {}
for name, path in candidate_paths.items():
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1024 / 1024
        model_sizes[name] = size_mb
        print(f'{name}: {size_mb:.2f} MB')

plt.figure(figsize=(8, 4))
plt.bar(model_sizes.keys(), model_sizes.values(), color=['#4C72B0', '#DD8452', '#55A868'])
plt.title('Model Size Comparison')
plt.ylabel('Size (MB)')
plt.tight_layout()
plt.savefig('../results/plots/model_sizes.png', dpi=150)
plt.show()

## 6. Deployment Options

### Option A: GitHub Pages (Free, Easiest)

```bash
# Commit the web/ folder
git add web/
git commit -m "Add web deployment files"
git push origin main
```

Then go to repository Settings → Pages → Source: main / `web/` folder.

Your app will be live at: `https://<username>.github.io/ASL/`

---

### Option B: Vercel / Netlify (Free Tier)

1. Connect your GitHub repository to [Vercel](https://vercel.com) or [Netlify](https://netlify.com).
2. Set Root Directory to `web/`.
3. Deploy — automatic updates on every push.

---

### Option C: Run Locally

```bash
# Start a local HTTP server (required for camera access)
cd web/
python -m http.server 8080
# Open http://localhost:8080 in Chrome
```

---

### Usage in Browser

1. Open the deployed URL.
2. Click **Start Camera** and allow camera permission.
3. Hold up an ASL hand sign in front of the camera.
4. The top-5 predicted letters and confidence scores update in real-time.
5. Click **Stop Camera** to end the session.


## Summary

The project is now fully deployed:
- Model converted to TensorFlow.js and placed in `web/tfjs_model/`
- Class labels exported to `web/labels.json`
- Web interface in `web/index.html` ready to use

For the thesis, the web deployment demonstrates real-world applicability of the trained model.
